# Sentinel-1 / C-SAR

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=sentinel/sentinel-1-c-sar). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1

Level 1: SLC (Single Look Complex, phase-preserved) or GRD (Ground Range Detected, amplitude-only, geocoded). SLC is required for InSAR; GRD is the more commonly distributed, browser-friendly product.

Searches the Microsoft Planetary Computer STAC catalogue (no account needed) for Sentinel-1 GRD scenes and plots VV backscatter in decibel. For terrain-corrected backscatter use the collection sentinel-1-rtc.

In [ ]:
%pip install -q pystac-client planetary-computer rioxarray matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import planetary_computer
import pystac_client
import rioxarray

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)


def open_overview(href, level=2):
    """Open a COG at reduced resolution (fast), falling back to full resolution."""
    try:
        return rioxarray.open_rasterio(href, overview_level=level, masked=True).squeeze()
    except Exception:
        return rioxarray.open_rasterio(href, masked=True).squeeze()

search = catalog.search(
    collections=["sentinel-1-grd"],
    bbox=[12.4, 41.8, 12.6, 42.0],                      # Rome - change to your area
    datetime="2024-06-01/2024-06-30",
)
items = search.item_collection()
if not items:
    raise SystemExit("No scenes found - widen the dates or the bounding box")
item = items[0]
print(len(items), "scenes found; using", item.id, item.datetime)
print("assets:", sorted(item.assets))

# GRD assets are in slant/ground-range geometry (not map-projected): plotted here in image coordinates
vv = open_overview(item.assets["vv"].href, level=3)
db = 10 * np.log10(vv.where(vv > 0))
db.plot(cmap="gray", vmin=-25, vmax=0)
plt.title("VV backscatter (dB) - " + item.id)
plt.show()

## L2

Level 2: OCN (Ocean) products — derived geophysical variables (wind, wave, current) for maritime applications, not general land InSAR.

Searches the Copernicus Data Space catalogue (anonymous). Set DOWNLOAD = True to fetch the first product.

In [ ]:
%pip install -q requests

In [ ]:
import requests

CATALOGUE = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
AREA = "POLYGON((12.0 41.0, 13.0 41.0, 13.0 42.5, 12.0 42.5, 12.0 41.0))"   # Italy (lon lat)
START, END = "2024-06-01T00:00:00.000Z", "2024-06-03T00:00:00.000Z"

flt = (
    f"Collection/Name eq 'SENTINEL-1' "
    f"and Attributes/OData.CSC.StringAttribute/any(a:a/Name eq 'productType' "
    f"and a/OData.CSC.StringAttribute/Value eq 'OCN') "
    f"and ContentDate/Start gt {START} and ContentDate/Start lt {END} "
    f"and OData.CSC.Intersects(area=geography'SRID=4326;{AREA}')"
)
response = requests.get(CATALOGUE, params={"$filter": flt, "$orderby": "ContentDate/Start asc", "$top": 5})
response.raise_for_status()
products = response.json()["value"]
print(len(products), "Sentinel-1 OCN products found")
for p in products:
    print(p["Name"], round(p["ContentLength"] / 1e6), "MB")

# Searching is anonymous. Downloading needs a free Copernicus Data Space account.
DOWNLOAD = False
if DOWNLOAD and products:
    import getpass
    token = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={"client_id": "cdse-public", "grant_type": "password",
              "username": input("CDSE e-mail: "), "password": getpass.getpass("CDSE password: ")},
    ).json()["access_token"]
    product = products[0]
    url = "https://download.dataspace.copernicus.eu/odata/v1/Products(%s)/$value" % product["Id"]
    with requests.get(url, headers={"Authorization": "Bearer " + token}, stream=True) as r:
        r.raise_for_status()
        with open(product["Name"] + ".zip", "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print("saved", product["Name"] + ".zip")

# After downloading, OCN files are NetCDF4: open them with xarray (xr.open_dataset) to read wind, wave and current fields.